# Notebook for retrieving Open-Meteo Weather Data


In [1]:
#imports
import requests
import json
import csv
import time


List of cities:

first run is major US cities + some from North America

In [ ]:

# ── City list ──────────────────────────────────────────────────────────────────
CITIES = [
    # USA
    "New York City",
    "Chicago",
    "Los Angeles",
    "Washington",
    "Dallas",
    "San Francisco",
    "Miami",
    "Seattle",
    "Boston",
    "Denver",
    "Houston",
    "Phoenix",
    "Portland",
    "Philadelphia",
    "Honolulu",
    "Salt Lake City",
    "Nashville",
    "Oklahoma City",
    "Las Vegas",
    "Baltimore",
    "Kansas City",
    "Omaha",
    "Minneapolis",
    "Cleveland",
    # Additional North American cities
    "Toronto",
    "Ottawa",
    "Halifax",
    "Mexico City",
    "Havana",
    "Montreal",
    "Tijuana",
    "Panama City",
]

In [ ]:

BASE_URL = "https://geocoding-api.open-meteo.com/v1/search"
FILEPATH_OUT= "location-list/" 

# ── Fetch geodata for a single city ───────────────────────────────────────────
def fetch_geodata(city_name: str, count: int = 1) -> dict | None:
    """
    Query the Open-Meteo geocoding API for `city_name`.
    Returns the top result as a flat dict, or None if nothing found.
    """
    params = {
        "name": city_name,
        "count": count,
        "language": "en",
        "format": "json",
    }
    response = requests.get(BASE_URL, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    results = data.get("results")
    if not results:
        print(f"  ⚠  No results for: {city_name}")
        return None

    top = results[0]
    return {
        "query":        city_name,
        "id":           top.get("id"),
        "name":         top.get("name"),
        "latitude":     top.get("latitude"),
        "longitude":    top.get("longitude"),
        "elevation":    top.get("elevation"),
        "country_code": top.get("country_code"),
        "country":      top.get("country"),
        "admin1":       top.get("admin1"),       # state / province
        "admin2":       top.get("admin2"),
        "timezone":     top.get("timezone"),
        "population":   top.get("population"),
    }


# ── Main routine ───────────────────────────────────────────────────────────────
def fetch_all_cities(
    cities: list[str],
    json_out: str = "geodata.json",
    csv_out:  str = "geodata.csv",
    delay:    float = 0.2,          # seconds between requests (be polite)
) -> list[dict]:
    """
    Iterate over `cities`, fetch geodata for each, then save to JSON and CSV.

    Parameters
    ----------
    cities   : list of city name strings
    json_out : output path for the JSON file
    csv_out  : output path for the CSV file
    delay    : pause between API requests (seconds)

    Returns
    -------
    List of result dicts (one per city that returned a hit).
    """
    results = []

    for city in cities:
        print(f"Fetching: {city} …")
        try:
            record = fetch_geodata(city)
            if record:
                results.append(record)
        except requests.RequestException as exc:
            print(f"  ✗ Request error for '{city}': {exc}")
        time.sleep(delay)

    # ── Save JSON ──────────────────────────────────────────────────────────────
    with open(json_out, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\n✓ JSON saved → {json_out}  ({len(results)} records)")

    # ── Save CSV ───────────────────────────────────────────────────────────────
    if results:
        fieldnames = list(results[0].keys())
        with open(csv_out, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(results)
        print(f"✓ CSV  saved → {csv_out}")

    return results


# ── Entry point ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    geodata = fetch_all_cities(
        cities=CITIES,
        json_out=f"{FILEPATH_OUT}geodata.json",  # ← f-string interpolates the variable
        csv_out=f"{FILEPATH_OUT}geodata.csv",    # ← f-string interpolates the variable
    )
    print(f"\nDone. {len(geodata)} cities geocoded.")    

Use the list generated in previous steps to get historical weather data from https://open-meteo.com/en/docs/historical-weather-api

In [ ]:
"""
fetch_weather_history.py
────────────────────────
Fetch daily historical weather (2000-01-01 → today) from the Open-Meteo
Historical Weather API for every city in the geocoding JSON produced by
geocode_cities.py.

Dependencies (install once):
    pip install openmeteo-requests requests-cache retry-requests pandas

Usage (in a notebook):
    # Set LOCATION_LIST_FILEPATH below, then run:
    %run fetch_weather_history.py
    # OR call directly:
    df = fetch_all_weather()
"""

import json
import time
from datetime import date, datetime
from pathlib import Path

import pandas as pd
import requests_cache
import openmeteo_requests
from retry_requests import retry

In [ ]:
#need to put cities list into 5 city chunks (5,000 calls per hour, 10,000 calls per day)
cities_1 = CITIES[0:5]
cities_2 = CITIES[5:10]
cities_3 = CITIES[10:15]
cities_4 = CITIES[15:20]
#can get up to cities_4 in by tomorrow (May 4th) probably

cities_5 = CITIES[20:25]
cities_6 = CITIES[25:30]
cities_7 = CITIES[30:35]
cities_8 = CITIES[35:40]




In [25]:
"""
fetch_weather_history.py
────────────────────────
Fetch daily historical weather (2000-01-01 → today) from the Open-Meteo
Historical Weather API for every city in the geocoding JSON produced by
geocode_cities.py.

Dependencies (install once):
    pip install openmeteo-requests requests-cache retry-requests pandas

Usage (in a notebook):
    # Set LOCATION_LIST_FILEPATH and CHUNK_NUMBER below, then run:
    %run fetch_weather_history.py
    # OR call directly:
    df = fetch_all_weather()
"""

import json
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests_cache
import openmeteo_requests
from retry_requests import retry

# ══════════════════════════════════════════════════════════════════════════════
# ── USER SETTINGS  ────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

# Path to the geodata JSON produced by geocode_cities.py
LOCATION_LIST_FILEPATH = "location-list/geodata.json"

# ── Chunking ───────────────────────────────────────────────────────────────────
# The full city list is split into chunks of CHUNK_SIZE.
# Set CHUNK_NUMBER to the chunk you want to run (1-indexed).
# e.g. with 32 cities and CHUNK_SIZE=5:
#   Chunk 1 → cities 1–5
#   Chunk 2 → cities 6–10
#   ...
#   Chunk 7 → cities 31–32
# Each chunk's output is saved separately, e.g. weather_history_chunk1.csv
# Run combine_chunks() at the end to merge all chunks into one file.


CHUNK_NUMBER = 6   # ← UPDATE THIS before each run
CHUNK_SIZE   = 5   # ← cities per chunk (keep at 5 to stay within API limits)

# Date range
START_DATE = "2000-01-01"
END_DATE   = date.today().isoformat()

# Output paths — chunk number is appended automatically
OUTPUT_DIR  = "weather-data"
OUTPUT_CSV  = f"{OUTPUT_DIR}/weather_history_chunk{CHUNK_NUMBER}.csv"
OUTPUT_JSON = f"{OUTPUT_DIR}/weather_history_chunk{CHUNK_NUMBER}.json"

# Final merged output (used by combine_chunks())
MERGED_CSV  = f"{OUTPUT_DIR}/weather_history.csv"
MERGED_JSON = f"{OUTPUT_DIR}/weather_history.json"

# How many cities to batch in a single API call
# With 14 variables x ~9,300 days, keep this at 1
BATCH_SIZE  = 1

# Seconds to pause between cities (61 = safe within per-minute limit)
BATCH_DELAY = 61

# ══════════════════════════════════════════════════════════════════════════════
# ── WEATHER VARIABLES  ────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

DAILY_VARIABLES = [
    "weather_code",                 # Weather Code (WMO)
    "temperature_2m_mean",          # Mean Temperature (2 m)
    "apparent_temperature_mean",    # Mean Apparent Temperature
    "rain_sum",                     # Rain Sum
    "precipitation_hours",          # Precipitation Hours
    "snowfall_sum",                 # Snowfall Sum
    "daylight_duration",            # Daylight Duration (s)
    "sunshine_duration",            # Sunshine Duration (s)
    "wind_gusts_10m_max",           # Maximum Wind Gusts
    "wind_speed_10m_mean",          # Mean Wind Speed
    "wind_gusts_10m_mean",          # Mean Wind Gusts
    "dew_point_2m_mean",            # Mean Dewpoint
    "cloud_cover_mean",             # Mean Cloud Cover
]

COL_RENAME = {
    "weather_code":              "weather_code",
    "temperature_2m_mean":       "temp_mean_c",
    "apparent_temperature_mean": "apparent_temp_mean_c",
    "rain_sum":                  "rain_sum_mm",
    "precipitation_hours":       "precipitation_hours",
    "snowfall_sum":              "snowfall_sum_cm",
    "daylight_duration":         "daylight_duration_s",
    "sunshine_duration":         "sunshine_duration_s",
    "wind_gusts_10m_max":        "wind_gusts_max_kmh",
    "wind_speed_10m_mean":       "wind_speed_mean_kmh",
    "wind_gusts_10m_mean":       "wind_gusts_mean_kmh",
    "dew_point_2m_mean":         "dewpoint_mean_c",
    "cloud_cover_mean":          "cloud_cover_mean_pct",
}

# ══════════════════════════════════════════════════════════════════════════════
# ── API CLIENT SETUP  ─────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def _make_client() -> openmeteo_requests.Client:
    cache_session = requests_cache.CachedSession(".om_cache", expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.4)
    return openmeteo_requests.Client(session=retry_session)

# ══════════════════════════════════════════════════════════════════════════════
# ── CORE FETCH LOGIC  ─────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def _fetch_batch(
    client: openmeteo_requests.Client,
    batch: list[dict],
    start_date: str,
    end_date: str,
) -> list[pd.DataFrame]:
    lats = [c["latitude"]  for c in batch]
    lons = [c["longitude"] for c in batch]

    params = {
        "latitude":        lats,
        "longitude":       lons,
        "start_date":      start_date,
        "end_date":        end_date,
        "daily":           DAILY_VARIABLES,
        "timezone":        "auto",
        "wind_speed_unit": "kmh",
    }

    url = "https://archive-api.open-meteo.com/v1/archive"
    responses = client.weather_api(url, params=params)

    frames = []
    for city_meta, resp in zip(batch, responses):
        daily = resp.Daily()

        dates = pd.date_range(
            start     = pd.to_datetime(daily.Time(),    unit="s", utc=True),
            end       = pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
            freq      = pd.Timedelta(seconds=daily.Interval()),
            inclusive = "left",
        ).date

        data = {"date": dates}
        for i, var_name in enumerate(DAILY_VARIABLES):
            data[var_name] = daily.Variables(i).ValuesAsNumpy()

        df = pd.DataFrame(data)
        df.rename(columns=COL_RENAME, inplace=True)

        df.insert(1, "city",      city_meta.get("query",    city_meta.get("name", "")))
        df.insert(2, "name",      city_meta.get("name",     ""))
        df.insert(3, "country",   city_meta.get("country",  ""))
        df.insert(4, "admin1",    city_meta.get("admin1",   ""))
        df.insert(5, "latitude",  city_meta.get("latitude",  resp.Latitude()))
        df.insert(6, "longitude", city_meta.get("longitude", resp.Longitude()))
        df.insert(7, "timezone",  resp.Timezone().decode() if isinstance(resp.Timezone(), bytes) else resp.Timezone())

        frames.append(df)

    return frames

# ══════════════════════════════════════════════════════════════════════════════
# ── PUBLIC ENTRY POINT  ───────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def fetch_all_weather(
    location_filepath: str = LOCATION_LIST_FILEPATH,
    chunk_number: int      = CHUNK_NUMBER,
    chunk_size: int        = CHUNK_SIZE,
    start_date: str        = START_DATE,
    end_date: str          = END_DATE,
    output_csv: str        = OUTPUT_CSV,
    output_json: str       = OUTPUT_JSON,
    batch_size: int        = BATCH_SIZE,
    batch_delay: float     = BATCH_DELAY,
) -> pd.DataFrame:
    # ── Load city list ─────────────────────────────────────────────────────────
    geo_path = Path(location_filepath)
    if not geo_path.exists():
        raise FileNotFoundError(
            f"Could not find geodata file at: {geo_path}\n"
            "Set LOCATION_LIST_FILEPATH to the path of your geodata.json."
        )

    with geo_path.open("r", encoding="utf-8") as f:
        all_cities = json.load(f)

    # ── Slice the chunk ────────────────────────────────────────────────────────
    total_chunks = (len(all_cities) + chunk_size - 1) // chunk_size
    if chunk_number < 1 or chunk_number > total_chunks:
        raise ValueError(f"CHUNK_NUMBER must be between 1 and {total_chunks} (got {chunk_number})")

    start_idx = (chunk_number - 1) * chunk_size
    end_idx   = start_idx + chunk_size
    cities    = all_cities[start_idx:end_idx]

    print(f"Total cities  : {len(all_cities)}")
    print(f"Chunk         : {chunk_number}/{total_chunks}  (cities {start_idx+1}–{min(end_idx, len(all_cities))})")
    print(f"Cities in run : {[c.get('query', c.get('name')) for c in cities]}")
    print(f"Date range    : {start_date} → {end_date}")
    print(f"Variables     : {len(DAILY_VARIABLES)}")
    print(f"Est. run time : ~{(len(cities) - 1) * batch_delay / 60:.1f} min\n")

    # Ensure output directory exists
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)

    client     = _make_client()
    all_frames = []

    total_batches = (len(cities) + batch_size - 1) // batch_size
    for batch_num, i in enumerate(range(0, len(cities), batch_size), start=1):
        batch      = cities[i : i + batch_size]
        city_names = [c.get("query", c.get("name", "?")) for c in batch]
        print(f"Batch {batch_num}/{total_batches}: {city_names}")

        try:
            frames = _fetch_batch(client, batch, start_date, end_date)
            all_frames.extend(frames)
            print(f"  ✓  {sum(len(f) for f in frames):,} rows fetched")
        except Exception as exc:
            print(f"  ✗  Error fetching batch {batch_num}: {exc}")

        if batch_num < total_batches:
            print(f"  ⏳ Waiting {batch_delay}s before next city...")
            time.sleep(batch_delay)

    if not all_frames:
        print("No data fetched.")
        return pd.DataFrame()

    combined = pd.concat(all_frames, ignore_index=True)
    combined.sort_values(["city", "date"], inplace=True)
    combined.reset_index(drop=True, inplace=True)

    print(f"\n✓ Total rows: {len(combined):,}")

    combined.to_csv(output_csv, index=False)
    print(f"✓ CSV  saved → {output_csv}")

    json_ready = combined.copy()
    json_ready["date"] = json_ready["date"].astype(str)
    json_ready.to_json(output_json, orient="records", indent=2, force_ascii=False)
    print(f"✓ JSON saved → {output_json}")

    return combined

# ══════════════════════════════════════════════════════════════════════════════
# ── COMBINE ALL CHUNKS  ───────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def combine_chunks(
    output_dir: str        = OUTPUT_DIR,
    merged_csv: str        = MERGED_CSV,
    merged_json: str       = MERGED_JSON,
    chunk_size: int        = CHUNK_SIZE,
    location_filepath: str = LOCATION_LIST_FILEPATH,
) -> pd.DataFrame:
    """
    Merge all per-chunk CSVs into a single weather_history.csv / .json.
    Run this after all chunks are complete.

    Usage:
        from fetch_weather_history import combine_chunks
        df = combine_chunks()
    """
    with open(location_filepath, "r") as f:
        all_cities = json.load(f)
    total_chunks = (len(all_cities) + chunk_size - 1) // chunk_size

    print(f"Looking for {total_chunks} chunk files in '{output_dir}/'...\n")

    frames = []
    for n in range(1, total_chunks + 1):
        path = Path(output_dir) / f"weather_history_chunk{n}.csv"
        if path.exists():
            df = pd.read_csv(path)
            frames.append(df)
            print(f"  ✓ Chunk {n}: {len(df):,} rows")
        else:
            print(f"  ⚠ Chunk {n} not found: {path}  (skipping)")

    if not frames:
        print("No chunk files found.")
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    combined.sort_values(["city", "date"], inplace=True)
    combined.reset_index(drop=True, inplace=True)

    combined.to_csv(merged_csv, index=False)
    print(f"\n✓ Merged CSV  → {merged_csv}  ({len(combined):,} total rows)")

    json_ready = combined.copy()
    json_ready["date"] = json_ready["date"].astype(str)
    json_ready.to_json(merged_json, orient="records", indent=2, force_ascii=False)
    print(f"✓ Merged JSON → {merged_json}")

    return combined

# ══════════════════════════════════════════════════════════════════════════════
# ── CLI / notebook entry point  ───────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    df = fetch_all_weather()
    if not df.empty:
        print("\nSample (first 5 rows):")
        print(df.head().to_string(index=False))

Total cities  : 32
Chunk         : 6/7  (cities 26–30)
Cities in run : ['Ottawa', 'Halifax', 'Mexico City', 'Havana', 'Montreal']
Date range    : 2000-01-01 → 2026-05-04
Variables     : 13
Est. run time : ~4.1 min

Batch 1/5: ['Ottawa']
  ✗  Error fetching batch 1: failed to request 'https://archive-api.open-meteo.com/v1/archive': {'error': True, 'reason': 'Daily API request limit exceeded. Please try again tomorrow.'}
  ⏳ Waiting 61s before next city...


KeyboardInterrupt: 

# FOR TOMORROW MAY 4:
- Chunk 1 done
- Chunk 2 done
- Chunk 3 done
- Chunk 4 is incomplete (only Salt Lake City)
- Chunk 5 done

- add to normalization
- add to website

In [26]:
# Combine weather chunk CSVs into a single CSV
from pathlib import Path
import pandas as pd

def combine_weather_chunks(folder=None):
    """Find all files matching weather_history_chunk*.csv in `folder`,
    concatenate them with pandas, and write to `combined/weather_history.csv`.

    Args:
        folder (str | Path | None): path to the `weather-data` folder. If None,
            defaults to the notebook-relative `weather-data` directory.

    Returns:
        pathlib.Path: path to the written combined CSV, or None if no files found.
    """
    if folder is None:
        # notebook is in 04_Data-Experience; default to the attached weather-data folder
        folder = Path("data_as_material/04_Data-Experience/weather-data")
    folder = Path(folder).expanduser()
    csv_paths = sorted(folder.glob("weather_history_chunk*.csv"))
    if not csv_paths:
        print(f"No chunk CSV files found in {folder}")
        return None
    dfs = [pd.read_csv(p) for p in csv_paths]
    combined = pd.concat(dfs, ignore_index=True)
    outdir = folder / "combined"
    outdir.mkdir(parents=True, exist_ok=True)
    outpath = outdir / "weather_history.csv"
    combined.to_csv(outpath, index=False)
    print(f"Wrote {len(combined)} rows to {outpath}")
    return outpath


In [27]:
combine_weather_chunks(folder="weather-data")


Wrote 202041 rows to weather-data/combined/weather_history.csv


PosixPath('weather-data/combined/weather_history.csv')

In [14]:
#import and export combined weather data to drop showers_sum which is 0 across the board
# df_remove_showers = pd.read_csv("weather-data/combined/weather_history.csv")
# df_remove_showers.drop(columns=["showers_sum_mm"], inplace=True)
# df_remove_showers.to_csv("weather-data/combined/weather_history.csv", index=False)


# Normalization

In [28]:
"""
normalize_weather.py

Prepares a raw weather CSV for KNN / Euclidean-distance search.

Steps:
  1. Engineer cleaner features from raw columns
  2. Bucket WMO weather_code into an ordinal severity scale
  3. log1p-transform right-skewed precipitation variables
  4. RobustScale all continuous features
  5. Apply perceptual weights so no single variable dominates
  6. Write normalized CSV (metadata columns kept but not scaled)

Usage:
    python normalize_weather.py --input raw.csv --output normalized.csv

    # optionally save the fitted scaler for reuse on live data:
    python normalize_weather.py --input raw.csv --output normalized.csv --save-scaler scaler.pkl
"""

import pickle
import sys

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler


# ---------------------------------------------------------------------------
# WMO weather code → ordinal severity bucket
# ---------------------------------------------------------------------------
# 0 = clear sky
# 1 = mostly clear / partly cloudy / overcast
# 2 = fog / depositing rime fog
# 3 = drizzle (light → dense)
# 4 = rain (slight → heavy)
# 5 = freezing rain / freezing drizzle
# 6 = snow (slight → heavy)
# 7 = rain/snow showers
# 8 = thunderstorm
WMO_BUCKETS = {
    0:  0,            # clear sky
    1:  1, 2:  1, 3:  1,          # mainly clear → overcast
    45: 2, 48: 2,                  # fog
    51: 3, 53: 3, 55: 3,           # drizzle
    56: 5, 57: 5,                  # freezing drizzle
    61: 4, 63: 4, 65: 4,           # rain
    66: 5, 67: 5,                  # freezing rain
    71: 6, 73: 6, 75: 6,           # snow
    77: 6,                         # snow grains
    80: 7, 81: 7, 82: 7,           # rain showers
    85: 7, 86: 7,                  # snow showers
    95: 8,                         # thunderstorm
    96: 8, 99: 8,                  # thunderstorm w/ hail
}

# ---------------------------------------------------------------------------
# Perceptual weights (applied after scaling by multiplying each column).
# Higher = more influence on Euclidean distance.
# Tune these based on qualitative testing.
# ---------------------------------------------------------------------------
WEIGHTS = {
    "temperature_2m_mean":      2.0,
    "apparent_temperature_mean": 1.5,
    "dewpoint_mean_c":          1.5,   # humidity / "stickiness"
    "rain_sum_log1p":           1.5,
    "snowfall_sum_log1p":       1.5,
    "precipitation_hours":      0.75,
    "sunshine_fraction":        1.0,
    "cloud_cover_mean_pct":     0.75,
    "wind_speed_mean_kmh":      0.75,
    "wind_gusts_max_kmh":       1.0,
    "weather_code_bucket":      1.0,
}


# ---------------------------------------------------------------------------
# Metadata columns — kept in output but never scaled
# ---------------------------------------------------------------------------
META_COLS = [
    "date", "city", "name", "country", "admin1",
    "latitude", "longitude", "timezone",
]


def load(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"Loaded {len(df):,} rows, {df.shape[1]} columns from '{path}'")
    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Derive cleaner features; drop raw columns that are redundant."""
    out = df[META_COLS].copy()

    # -- Weather code bucket (ordinal, 0–8) ---------------------------------
    out["weather_code_bucket"] = (
        df["weather_code"]
        .fillna(0)
        .astype(int)
        .map(WMO_BUCKETS)
        .fillna(1)          # unknown codes → 'mostly clear'
        .astype(float)
    )

    # -- Temperature / humidity ---------------------------------------------
    out["temperature_2m_mean"]       = df["temp_mean_c"]
    out["apparent_temperature_mean"] = df["apparent_temp_mean_c"]
    out["dewpoint_mean_c"]           = df["dewpoint_mean_c"]

    # -- Precipitation (log1p to compress right skew) -----------------------
    out["rain_sum_log1p"]      = np.log1p(df["rain_sum_mm"].clip(lower=0))
    out["snowfall_sum_log1p"]  = np.log1p(df["snowfall_sum_cm"].clip(lower=0))
    out["precipitation_hours"] = df["precipitation_hours"].clip(lower=0)

    # -- Sunshine fraction (sunshine / daylight, avoids lat/season bias) ----
    daylight  = df["daylight_duration_s"].replace(0, np.nan)
    sunshine  = df["sunshine_duration_s"].clip(lower=0)
    out["sunshine_fraction"] = (sunshine / daylight).clip(0, 1).fillna(0)

    # -- Cloud cover --------------------------------------------------------
    out["cloud_cover_mean_pct"] = df["cloud_cover_mean_pct"].clip(0, 100)

    # -- Wind ---------------------------------------------------------------
    out["wind_speed_mean_kmh"] = df["wind_speed_mean_kmh"].clip(lower=0)
    out["wind_gusts_max_kmh"]  = df["wind_gusts_max_kmh"].clip(lower=0)
    # drop wind_gusts_mean_kmh — correlated with the other two

    return out


def scale_and_weight(df: pd.DataFrame, scaler=None):
    """
    RobustScale continuous features, then apply perceptual weights.

    Returns (normalized_df, fitted_scaler).
    Pass a pre-fitted scaler to transform live data without refitting.
    """
    feature_cols = list(WEIGHTS.keys())
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected feature columns: {missing}")

    X = df[feature_cols].values.astype(float)

    if scaler is None:
        scaler = RobustScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = scaler.transform(X)

    # Apply weights by multiplying each column
    weight_vector = np.array([WEIGHTS[c] for c in feature_cols])
    X_weighted = X_scaled * weight_vector

    scaled_df = pd.DataFrame(X_weighted, columns=feature_cols, index=df.index)
    return scaled_df, scaler


def normalize(input_path: str, output_path: str, scaler_path: str | None = None):
    df_raw = load(input_path)

    print("Engineering features...")
    df_eng = engineer_features(df_raw)

    print("Scaling and weighting...")
    df_scaled, scaler = scale_and_weight(df_eng)

    # Combine metadata + normalized features
    df_out = pd.concat([df_eng[META_COLS].reset_index(drop=True),
                        df_scaled.reset_index(drop=True)], axis=1)

    df_out.to_csv(output_path, index=False)
    print(f"Normalized CSV written to '{output_path}' ({len(df_out):,} rows)")

    if scaler_path:
        with open(scaler_path, "wb") as f:
            pickle.dump({"scaler": scaler, "feature_cols": list(WEIGHTS.keys()),
                         "weights": WEIGHTS}, f)
        print(f"Scaler saved to '{scaler_path}'")

    return df_out, scaler


def normalize_live(row: dict, scaler_path: str) -> np.ndarray:
    """
    Convenience function to normalize a single live weather observation
    using a previously fitted scaler.

    row: dict of raw weather values (same schema as the CSV)
    Returns: 1-D numpy array ready for KNN distance computation
    """
    with open(scaler_path, "rb") as f:
        bundle = pickle.load(f)

    scaler      = bundle["scaler"]
    feature_cols = bundle["feature_cols"]
    weights     = bundle["weights"]

    df_single = engineer_features(pd.DataFrame([row]))
    X = df_single[feature_cols].values.astype(float)
    X_scaled = scaler.transform(X)
    weight_vector = np.array([weights[c] for c in feature_cols])
    return (X_scaled * weight_vector)[0]



In [34]:
df_normalized, scaler = normalize(
    input_path="weather-data/combined/weather_history.csv",
    output_path="weather-data/normalized/combined_weather_normalized.csv",
    scaler_path="scaler.pkl",
)

Loaded 202,041 rows, 22 columns from 'weather-data/combined/weather_history.csv'
Engineering features...
Scaling and weighting...
Normalized CSV written to 'weather-data/normalized/combined_weather_normalized.csv' (202,041 rows)
Scaler saved to 'scaler.pkl'


# Fetch current weather data and normalize to vector space

In [37]:
"""
fetch_today_weather.py

Fetches today's weather for a given location from Open-Meteo,
normalizes it using the saved scaler from normalize_weather.py,
and returns a vector ready for KNN distance comparison.

Usage (notebook cell):
    from fetch_today_weather import fetch_and_normalize

    vector, df_raw, df_normalized = fetch_and_normalize(
        scaler_path="scaler.pkl",
        lat=40.7128,
        lon=-74.0060,
        timezone="America/New_York",
    )
"""

import pickle
from datetime import date

import numpy as np
import pandas as pd
import requests


# ---------------------------------------------------------------------------
# Open-Meteo API config
# ---------------------------------------------------------------------------
BASE_URL = "https://api.open-meteo.com/v1/forecast"

# Daily variables — split into standard and "additional" because Open-Meteo
# requires them all in the same &daily= param but documents them separately.
DAILY_PARAMS = ",".join([
    "weather_code",
    "temperature_2m_mean",
    "apparent_temperature_mean",
    "rain_sum",
    "precipitation_hours",
    "snowfall_sum",
    "daylight_duration",
    "sunshine_duration",
    "wind_gusts_10m_max",
    "wind_speed_10m_mean",
    "wind_gusts_10m_mean",   # additional daily variable
    "dew_point_2m_mean",     # additional daily variable
    "cloud_cover_mean",      # additional daily variable
])


def fetch_today(lat: float, lon: float, timezone: str) -> dict:
    """Fetch today's daily weather from Open-Meteo. Returns the raw daily dict."""
    params = {
        "latitude":     lat,
        "longitude":    lon,
        "daily":        DAILY_PARAMS,
        "timezone":     timezone,
        "forecast_days": 1,
    }

    resp = requests.get(BASE_URL, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    if "error" in data:
        raise ValueError(f"Open-Meteo error: {data.get('reason', data)}")

    # daily values come as lists of length 1 — unwrap to scalars
    daily = {k: v[0] for k, v in data["daily"].items()}
    daily["date"] = daily.pop("time")   # rename to match CSV schema
    return daily


def to_csv_schema(raw: dict, lat: float, lon: float, timezone: str) -> pd.DataFrame:
    """
    Remap Open-Meteo field names to the column names used in the historical CSV
    so that engineer_features() works identically on live and historical data.

    Historical CSV columns that matter for feature engineering:
        weather_code, temp_mean_c, apparent_temp_mean_c,
        rain_sum_mm, precipitation_hours, snowfall_sum_cm,
        daylight_duration_s, sunshine_duration_s,
        wind_gusts_max_kmh, wind_speed_mean_kmh, wind_gusts_mean_kmh,
        dewpoint_mean_c, cloud_cover_mean_pct
    """
    row = {
        # metadata (engineer_features expects these)
        "date":                  raw["date"],
        "city":                  "live",
        "name":                  "live",
        "country":               "live",
        "admin1":                "live",
        "latitude":              lat,
        "longitude":             lon,
        "timezone":              timezone,

        # weather variables — renamed to match historical CSV schema
        "weather_code":          raw["weather_code"],
        "temp_mean_c":           raw["temperature_2m_mean"],
        "apparent_temp_mean_c":  raw["apparent_temperature_mean"],
        "rain_sum_mm":           raw["rain_sum"],
        "precipitation_hours":   raw["precipitation_hours"],
        "snowfall_sum_cm":       raw["snowfall_sum"],
        "daylight_duration_s":   raw["daylight_duration"],
        "sunshine_duration_s":   raw["sunshine_duration"],
        "wind_gusts_max_kmh":    raw["wind_gusts_10m_max"],
        "wind_speed_mean_kmh":   raw["wind_speed_10m_mean"],
        "wind_gusts_mean_kmh":   raw["wind_gusts_10m_mean"],
        "dewpoint_mean_c":       raw["dew_point_2m_mean"],
        "cloud_cover_mean_pct":  raw["cloud_cover_mean"],

        # showers_sum_mm is always 0 in practice but engineer_features expects it
        "showers_sum_mm":        0.0,
    }
    return pd.DataFrame([row])


def fetch_and_normalize(
    scaler_path: str,
    lat: float = 40.7128,
    lon: float = -74.0060,
    timezone: str = "America/New_York",
) -> tuple[np.ndarray, pd.DataFrame, pd.DataFrame]:
    """
    Main entry point.

    Returns:
        vector       — 1-D numpy array ready for KNN distance computation
        df_raw       — single-row DataFrame in the historical CSV schema (pre-normalization)
        df_normalized — single-row DataFrame with all scaled feature columns
    """
    print(f"Fetching today's weather ({date.today()}) for lat={lat}, lon={lon}...")
    raw = fetch_today(lat, lon, timezone)
    print(f"  weather_code={raw['weather_code']}, "
          f"temp={raw['temperature_2m_mean']:.1f}°C, "
          f"rain={raw['rain_sum']:.1f}mm, "
          f"snow={raw['snowfall_sum']:.1f}cm")

    df_raw = to_csv_schema(raw, lat, lon, timezone)

    # Load the fitted scaler from the offline normalization run
    with open(scaler_path, "rb") as f:
        bundle = pickle.load(f)

    scaler       = bundle["scaler"]
    feature_cols = bundle["feature_cols"]

    # Apply the same feature engineering as the historical pipeline
    df_eng = engineer_features(df_raw)

    # Transform with the EXISTING scaler (no refit)
    df_scaled, _ = scale_and_weight(df_eng, scaler=scaler)

    df_normalized = pd.concat(
        [df_eng[["date", "city", "latitude", "longitude"]].reset_index(drop=True),
         df_scaled.reset_index(drop=True)],
        axis=1,
    )

    vector = df_scaled[feature_cols].values[0]

    print(f"Normalized vector ({len(vector)} features):")
    for name, val in zip(feature_cols, vector):
        print(f"  {name:<30} {val:+.4f}")

    return vector, df_raw, df_normalized

In [38]:


vector, df_raw, df_normalized = fetch_and_normalize(
    scaler_path="scaler.pkl",   # from your normalize_weather run
    lat=40.7128,
    lon=-74.0060,
    timezone="America/New_York",
)

Fetching today's weather (2026-05-04) for lat=40.7128, lon=-74.006...
  weather_code=1, temp=15.5°C, rain=0.0mm, snow=0.0cm
Normalized vector (11 features):
  temperature_2m_mean            +0.0271
  apparent_temperature_mean      -0.1418
  dewpoint_mean_c                -0.5662
  rain_sum_log1p                 +0.0000
  snowfall_sum_log1p             +0.0000
  precipitation_hours            +0.0000
  sunshine_fraction              +0.0005
  cloud_cover_mean_pct           -0.7049
  wind_speed_mean_kmh            +0.6246
  wind_gusts_max_kmh             +0.4541
  weather_code_bucket            +0.0000


In [39]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
NORMALIZED_CSV  = "weather-data/normalized/combined_weather_normalized.csv"
TOP_N           = 20

# Must match WEIGHTS keys in normalize_weather.py (showers dropped)
FEATURE_COLS = [
    "temperature_2m_mean",
    "apparent_temperature_mean",
    "dewpoint_mean_c",
    "rain_sum_log1p",
    "snowfall_sum_log1p",
    "precipitation_hours",
    "sunshine_fraction",
    "cloud_cover_mean_pct",
    "wind_speed_mean_kmh",
    "wind_gusts_max_kmh",
    "weather_code_bucket",
]

META_COLS = ["date", "city", "latitude", "longitude"]

# ---------------------------------------------------------------------------
# Load historical data
# ---------------------------------------------------------------------------
print("Loading historical normalized data...")
df_hist = pd.read_csv(NORMALIZED_CSV)

# Drop showers column if present (removed from feature set)
df_hist = df_hist.drop(columns=["showers_sum_log1p"], errors="ignore")

X_hist = df_hist[FEATURE_COLS].values.astype(float)

# ---------------------------------------------------------------------------
# Build BallTree index (Euclidean / l2 metric)
# ---------------------------------------------------------------------------
print(f"Building BallTree index over {len(df_hist):,} historical days...")
tree = BallTree(X_hist, metric="euclidean")

# ---------------------------------------------------------------------------
# Query — `vector` comes from fetch_and_normalize()
# ---------------------------------------------------------------------------
query = vector.reshape(1, -1)   # vector from fetch_today_weather.py

distances, indices = tree.query(query, k=TOP_N)
distances = distances[0]
indices   = indices[0]

# ---------------------------------------------------------------------------
# Build results table
# ---------------------------------------------------------------------------
results = df_hist.iloc[indices][META_COLS + FEATURE_COLS].copy()
results.insert(0, "rank", range(1, TOP_N + 1))
results.insert(2, "distance", distances.round(4))

print(f"\nTop {TOP_N} most similar historical days to today:\n")
print(results[["rank", "date", "city", "distance"]].to_string(index=False))

Loading historical normalized data...
Building BallTree index over 202,041 historical days...

Top 20 most similar historical days to today:

 rank       date          city  distance
    1 2009-03-03         Miami    0.2185
    2 2020-12-10        Dallas    0.2319
    3 2010-10-30        Dallas    0.2383
    4 2018-10-16      Portland    0.2420
    5 2018-04-27   Kansas City    0.2833
    6 2022-02-11        Dallas    0.2870
    7 2006-01-02        Dallas    0.2970
    8 2023-06-11   Minneapolis    0.2979
    9 2025-03-05       Houston    0.3004
   10 2019-01-10         Miami    0.3129
   11 2011-11-17       Houston    0.3294
   12 2020-02-28         Miami    0.3346
   13 2021-03-15        Dallas    0.3370
   14 2022-09-29         Omaha    0.3439
   15 2001-10-06        Dallas    0.3443
   16 2004-05-29 New York City    0.3476
   17 2001-05-15  Philadelphia    0.3501
   18 2006-11-22         Miami    0.3557
   19 2022-06-04       Toronto    0.3584
   20 2024-05-30       Toronto    0.35

# NYTimes API test

In [66]:
import requests
from datetime import datetime, timedelta
import json

# ── Fill these in ──────────────────────────────────────────────
API_KEY  = "dknXw2sJNp9CQhz1RFq00SlB3JsSXSDcBB2dOac96mUaXGOe"
DATE     = "2025-03-05"       # YYYY-MM-DD
LOCATION = "Dallas"     # City name as NYT knows it
# ──────────────────────────────────────────────────────────────

# Settings
DATE_WINDOW_DAYS = 7          # articles publish 1-2 days after events
MAX_PAGES        = 3          # 10 results per page, 3 pages = up to 30 results
RETURN_FIELDS    = "headline,pub_date,snippet,web_url,section_name,keywords"

# Build date range
start = datetime.strptime(DATE, "%Y-%m-%d")
end   = start + timedelta(days=DATE_WINDOW_DAYS)
begin_date_str = start.strftime("%Y%m%d")
end_date_str   = end.strftime("%Y%m%d")

BASE_URL = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

def query_nyt(page=0):
    params = {
        "begin_date": begin_date_str,
        "end_date":   end_date_str,
        "fq":         f'timesTag.location:("{LOCATION}")',
        "sort":       "relevance",
        "fl":         RETURN_FIELDS,
        "page":       page,
        "api-key":    API_KEY,
    }
    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()
    return r.json()
# Run query and collect results
all_articles = []
total_hits   = None

for page in range(MAX_PAGES):
    data = query_nyt(page=page)

    # ── Debug: inspect raw response if something looks wrong ──
    if "response" not in data or "metadata" not in data.get("response", {}):
        print("Unexpected response structure:")
        print(json.dumps(data, indent=2))
        break

    meta = data["response"]["metadata"]

    if total_hits is None:
        total_hits = meta["hits"]
        print(f"Found {total_hits} total hits for {LOCATION} around {DATE}")
        print(f"Fetching up to {MAX_PAGES * 10} results ({MAX_PAGES} pages)\n")

    articles = data["response"]["docs"]
    if not articles:
        print("No more articles.")
        break
    all_articles.extend(articles)

    
# Display results
for i, article in enumerate(all_articles, 1):
    headline = article.get("headline", {}).get("main", "No headline")
    pub_date = article.get("pub_date", "")[:10]
    snippet  = article.get("snippet", "")
    url      = article.get("web_url", "")
    section  = article.get("section_name", "")

    print(f"[{i}] {pub_date} | {section}")
    print(f"    {headline}")
    print(f"    {snippet}")
    print(f"    {url}")
    print()

Found 0 total hits for Dallas around 2025-03-05
Fetching up to 30 results (3 pages)

No more articles.


In [61]:
import requests
from datetime import datetime, timedelta
import json

# ── Diagnostics: test different fq filter approaches ──────────────────────

API_KEY  = "dknXw2sJNp9CQhz1RFq00SlB3JsSXSDcBB2dOac96mUaXGOe"
DATE     = "2025-01-01"
LOCATION = "New York City"
BASE_URL = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

# Build date range
start = datetime.strptime(DATE, "%Y-%m-%d")
end   = start + timedelta(days=7)
begin_date_str = start.strftime("%Y%m%d")
end_date_str   = end.strftime("%Y%m%d")

# Test different filter approaches
# NOTE: begin_date and end_date are SEPARATE query parameters, not part of fq
test_filters = [
    ("No filter (baseline)", ""),
    ("timesTag.location NYC", f'timesTag.location:("{LOCATION}")'),
    ("glocations NYC", f'glocations:("{LOCATION}")'),
]

for label, fq_value in test_filters:
    print(f"\n{'='*60}")
    print(f"Testing: {label}")
    print(f"Date range: {begin_date_str} → {end_date_str}")
    print(f"Filter (fq): {fq_value if fq_value else '(none)'}")
    print('='*60)
    
    params = {
        "begin_date": begin_date_str,
        "end_date":   end_date_str,
        "sort":       "relevance",
        "fl":         "headline,pub_date,snippet",
        "page":       0,
        "api-key":    API_KEY,
    }
    
    # fq is for content filtering only (location, type, etc)
    # begin_date and end_date are separate parameters
    if fq_value:
        params["fq"] = fq_value
    
    try:
        r = requests.get(BASE_URL, params=params)
        r.raise_for_status()
        data = r.json()
        
        if "response" in data and "metadata" in data["response"]:
            hits = data["response"]["metadata"]["hits"]
            print(f"✓ Success! Found {hits} hits")
            if hits > 0:
                doc = data["response"]["docs"][0]
                print(f"  Sample headline: {doc['headline']['main']}")
        else:
            print(f"✗ Unexpected response structure")
    except Exception as e:
        print(f"✗ Error: {e}")


Testing: No filter (baseline)
Date range: 20250101 → 20250108
Filter (fq): (none)
✓ Success! Found 790 hits
  Sample headline: Beet Salad With Celery and Pomegranate

Testing: timesTag.location NYC
Date range: 20250101 → 20250108
Filter (fq): timesTag.location:("New York City")
✓ Success! Found 36 hits
  Sample headline: A Sluggish Start for Congestion Pricing

Testing: glocations NYC
Date range: 20250101 → 20250108
Filter (fq): glocations:("New York City")
✓ Success! Found 0 hits
